In [2]:
!pip install -q transformers>=4.45.0 "trl==0.12.0" peft>=0.13.0 \
    bitsandbytes>=0.44.0 accelerate>=1.0.0 datasets huggingface_hub openai

In [3]:
import shutil, os
from google.colab import drive

if os.path.exists('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive')

drive.mount('/content/drive')
print("Google Drive mounted.")


Mounted at /content/drive
Google Drive mounted.


In [4]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in to HuggingFace via Colab Secrets")
except Exception:
    hf_token = os.environ.get("HF_TOKEN", "")
    if hf_token:
        login(token=hf_token)
        print("Logged in via env var")
    else:
        print("⚠️ WARNING: No HF_TOKEN found! Add it via: Colab left panel -> 🔑 Secrets -> add HF_TOKEN")

Logged in to HuggingFace via Colab Secrets


In [5]:
import os
import json
import torch
from pathlib import Path

# ============================================================
# USER CONFIG — edit these as needed
# ============================================================
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "/content/response_generator_model"      # Local Colab path
DRIVE_SAVE_DIR = "/content/drive/MyDrive/response_generator"
# Checkpoint save path on Drive (for resume after disconnect)
DRIVE_CKPT_DIR = "/content/drive/MyDrive/response_generator"
HUB_REPO = None

# Data paths
DATA_DIR = "/content/drive/MyDrive"
TRAIN_FILE = f"{DATA_DIR}/response_generator_train.jsonl"
TEST_FILE = f"{DATA_DIR}/response_generator_test.jsonl"

# Training hyperparameters
NUM_EPOCHS = 2
BATCH_SIZE = 4
GRAD_ACCUM = 4
LEARNING_RATE = 2e-4
MAX_SEQ_LEN = 512
WARMUP_RATIO = 0.05
LOGGING_STEPS = 50
SAVE_STEPS = 500
EVAL_STEPS = 500

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Verify environment
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} | VRAM: {vram:.1f} GB")
else:
    raise RuntimeError("No GPU detected!")

print(f"Checkpoints will be saved to: {DRIVE_CKPT_DIR}")

GPU: NVIDIA A100-SXM4-80GB | VRAM: 85.1 GB
Checkpoints will be saved to: /content/drive/MyDrive/response_generator


In [6]:
from google.colab import files
uploaded = files.upload()

Saving doctor_schedules.json to doctor_schedules.json


In [7]:
import re, time
from openai import OpenAI
from google.colab import userdata

VALID_DEPARTMENTS = {
    'Cardiology', 'Neurology', 'Dermatology', 'Gastroenterology',
    'Endocrinology', 'Pulmonology', 'Infectious Disease',
    'Orthopedics', 'Urology', 'General Medicine',
}
VALID_URGENCIES = {'Routine', 'Urgent', 'Emergency'}
DEPT_LIST = ', '.join(sorted(VALID_DEPARTMENTS))

# ── DeepSeek (primary) ──
deepseek_client = OpenAI(
    api_key=userdata.get('DEEPSEEK_API_KEY'),
    base_url='https://api.deepseek.com',
)

# ── ChatGPT (fallback) ──
chatgpt_client = OpenAI(
    api_key=userdata.get('OPENAI_API_KEY'),
)

CLASSIFY_SYSTEM_PROMPT = f'''You are a medical triage classifier.
Given a patient's description, do TWO things:
1. Classify into exactly one department from: {DEPT_LIST}
   If the patient's issue does not fit ANY of these departments, output Department: SKIP
2. Assign urgency: Routine, Urgent, or Emergency

Respond in exactly this format (nothing else):
Department: <department or SKIP>
Urgency: <Routine|Urgent|Emergency>'''


def _call_llm(client, model, patient_text):
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': CLASSIFY_SYSTEM_PROMPT},
            {'role': 'user', 'content': patient_text},
        ],
        max_tokens=50,
        temperature=0.1,
    )
    return resp.choices[0].message.content.strip()


def classify_patient_text(patient_text):
    """DeepSeek first, fallback to ChatGPT. Returns (dept, urgency) or (None, None) for SKIP."""
    text = None
    try:
        text = _call_llm(deepseek_client, 'deepseek-chat', patient_text)
    except Exception as e:
        print(f'  [DeepSeek failed: {e}], trying ChatGPT...')

    if text is None:
        try:
            text = _call_llm(chatgpt_client, 'gpt-4o-mini', patient_text)
        except Exception as e:
            print(f'  [ChatGPT also failed: {e}]')
            return None, None

    dept_match = re.search(r'Department:\s*([^\n]+)', text)
    urg_match = re.search(r'Urgency:\s*([^\n]+)', text)
    department = dept_match.group(1).strip() if dept_match else None
    urgency = urg_match.group(1).strip() if urg_match else 'Routine'

    if department is None or department == 'SKIP' or department not in VALID_DEPARTMENTS:
        return None, None
    if urgency not in VALID_URGENCIES:
        urgency = 'Routine'
    return department, urgency


print('API clients configured.')
print(f'Valid departments: {DEPT_LIST}')

API clients configured.
Valid departments: Cardiology, Dermatology, Endocrinology, Gastroenterology, General Medicine, Infectious Disease, Neurology, Orthopedics, Pulmonology, Urology


In [8]:
import json, random, sys, time
from collections import Counter
from datasets import load_dataset as dl

# Load doctor schedules (uploaded in previous cell)
with open("doctor_schedules.json") as f:
    schedules = json.load(f)

schedules_by_dept = {}
for entry in schedules:
    schedules_by_dept.setdefault(entry['department'], []).append(entry)
print(f"Loaded {len(schedules)} schedule entries, {len(schedules_by_dept)} departments")

INSTRUCTION_TEMPLATE = (
    "You are a compassionate medical assistant. "
    "A patient has been assigned an appointment. "
    "Write a warm, clear appointment confirmation and practical pre-visit instructions. "
    "Keep the tone professional but reassuring. "
    "Format your response as:\n"
    "Confirmation: <one sentence confirming the appointment>\n"
    "Instructions: <2-4 specific pre-visit instructions>"
)

MIN_RESPONSE_LEN = 80
MAX_RESPONSE_LEN = 800
MAX_SYMPTOM_LEN = 200


def make_record(patient_q, doctor_a, department, urgency):
    """Build a training record using LLM API classification result."""
    doctor_a = str(doctor_a).strip()
    if len(doctor_a) < MIN_RESPONSE_LEN or len(doctor_a) > MAX_RESPONSE_LEN:
        return None

    dept_scheds = schedules_by_dept.get(department,
                      schedules_by_dept.get('General Medicine', schedules))
    entry = random.choice(dept_scheds)
    doctor = entry['doctor']
    time_slot = f"{entry['day']} at {entry['time_slot']}"
    symptoms = str(patient_q).strip()[:MAX_SYMPTOM_LEN]

    formatted_output = (
        f'Confirmation: Your appointment with {doctor} in {department} '
        f'has been confirmed for {time_slot}. '
        f"We're here to help with your concerns.\n"
        f'Instructions: {doctor_a}'
    )

    return {
        'instruction': INSTRUCTION_TEMPLATE,
        'input': (f'Patient symptoms: {symptoms}\n'
                  f'Assigned department: {department}\n'
                  f'Doctor: {doctor}\n'
                  f'Appointment: {time_slot}\n'
                  f'Urgency: {urgency}'),
        'output': formatted_output,
        'department': department,
        'urgency': urgency,
    }


# ============================================================
# Full data generation: 20% sample, labeled by DeepSeek API
# Supports resume: saves progress to Drive every 1000 records
# ============================================================
PROGRESS_FILE = "/content/drive/MyDrive/response_generator_progress.jsonl"
SAVE_EVERY = 1000

# Check for existing progress to resume from
records = []
skipped = 0
start_from = 0

if os.path.exists(PROGRESS_FILE):
    with open(PROGRESS_FILE) as f:
        for line in f:
            obj = json.loads(line)
            if obj.get('_type') == '_meta':
                start_from = obj['next_index']
                skipped = obj['skipped']
            else:
                records.append(obj)
    print(f"Resumed from progress file: {len(records)} records, skipped={skipped}, continuing from index {start_from}")
else:
    print("No progress file found. Starting from scratch.")

random.seed(42)
ds = dl("ruslanmv/ai-medical-chatbot", split="train")
print(f"Total rows in dataset: {len(ds)}")

n_sample = int(len(ds) * 0.10)
indices = random.sample(range(len(ds)), n_sample)
sampled = ds.select(indices)
total = len(sampled)
remaining = total - start_from
print(f"Sampled: {total} rows, processing {remaining} remaining from index {start_from}")


def save_progress(records, next_index, skipped):
    """Save current progress to Drive for resume after disconnect."""
    with open(PROGRESS_FILE, 'w') as f:
        for r in records:
            f.write(json.dumps(r) + '\n')
        f.write(json.dumps({'_type': '_meta', 'next_index': next_index, 'skipped': skipped}) + '\n')


def fmt_time(seconds):
    """Format seconds to mm:ss or hh:mm:ss."""
    if seconds < 0:
        return "--:--"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"{h}:{m:02d}:{s:02d}" if h else f"{m}:{s:02d}"


t0 = time.time()

for i in range(start_from, total):
    row = sampled[i]
    patient_text = row.get('Patient', row.get('input', ''))
    doctor_text = row.get('Doctor', row.get('output', ''))

    # DeepSeek API classification (fallback to ChatGPT)
    department, urgency = classify_patient_text(str(patient_text).strip()[:500])

    if department is None:
        skipped += 1
    else:
        rec = make_record(patient_text, doctor_text, department, urgency)
        if rec:
            records.append(rec)

    # Save progress every SAVE_EVERY records
    if (i + 1) % SAVE_EVERY == 0:
        save_progress(records, i + 1, skipped)

    # Progress display every 50 records
    done = i + 1 - start_from
    if done % 50 == 0 or (i + 1) == total:
        elapsed = time.time() - t0
        speed = done / elapsed if elapsed > 0 else 0
        eta = (remaining - done) / speed if speed > 0 else 0
        pct = (i + 1) / total
        bar = chr(9608) * int(pct * 30) + chr(9617) * (30 - int(pct * 30))
        saved_flag = " [saved]" if (i + 1) % SAVE_EVERY == 0 else ""
        print(f'\r  {bar} {i+1}/{total} ({pct:.0%}) | kept={len(records)} skip={skipped} | {speed:.1f} it/s | ETA: {fmt_time(eta)}{saved_flag}', end='')
        sys.stdout.flush()

# Final save
save_progress(records, total, skipped)
elapsed_total = time.time() - t0

print(f'\n\nDone in {fmt_time(elapsed_total)}')
print(f'Total records: {len(records)}')
print(f'Skipped (not in 10 departments): {skipped}')

# Department distribution
dept_counter = Counter(r['department'] for r in records)
print('\n--- Department Distribution ---')
for dept, cnt in sorted(dept_counter.items(), key=lambda x: -x[1]):
    print(f'  {dept:<25} {cnt:>5}  ({cnt/len(records):.1%})')

# Train / Test split and save
random.shuffle(records)
split = int(len(records) * 0.85)
train, test = records[:split], records[split:]

for name, data in [("train", train), ("test", test)]:
    path = f"/content/drive/MyDrive/response_generator_{name}.jsonl"
    with open(path, "w") as f:
        for r in data:
            f.write(json.dumps(r) + "\n")

print(f"\nTrain: {len(train)} | Test: {len(test)} — saved to Drive")

# Clean up progress file (data generation complete)
if os.path.exists(PROGRESS_FILE):
    os.remove(PROGRESS_FILE)
    print("Progress file cleaned up.")

Loaded 824 schedule entries, 10 departments
Resumed from progress file: 3789 records, skipped=403, continuing from index 5000


README.md:   0%|          | 0.00/863 [00:00<?, ?B/s]

dialogues.parquet:   0%|          | 0.00/142M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/256916 [00:00<?, ? examples/s]

Total rows in dataset: 256916
Sampled: 25691 rows, processing 20691 remaining from index 5000
  ██████████████████████████████ 25691/25691 (100%) | kept=19535 skip=2166 | 0.7 it/s | ETA: 0:00

Done in 8:02:12
Total records: 19535
Skipped (not in 10 departments): 2166

--- Department Distribution ---
  Dermatology                2579  (13.2%)
  General Medicine           2525  (12.9%)
  Gastroenterology           2365  (12.1%)
  Urology                    2104  (10.8%)
  Neurology                  2041  (10.4%)
  Orthopedics                2033  (10.4%)
  Endocrinology              1850  (9.5%)
  Cardiology                 1707  (8.7%)
  Pulmonology                1316  (6.7%)
  Infectious Disease         1015  (5.2%)

Train: 16604 | Test: 2931 — saved to Drive
Progress file cleaned up.


In [9]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "test": TEST_FILE})

print(f"Train: {len(dataset['train'])} samples | Test: {len(dataset['test'])} samples")
print(f"Columns: {dataset['train'].column_names}")

print(f"\n--- Sample Data Preview ---")
sample = dataset['train'][0]
for k in ['instruction', 'input', 'output']:
    print(f"[{k}]: {sample[k][:200]}...")

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train: 16604 samples | Test: 2931 samples
Columns: ['instruction', 'input', 'output', 'department', 'urgency']

--- Sample Data Preview ---
[instruction]: You are a compassionate medical assistant. A patient has been assigned an appointment. Write a warm, clear appointment confirmation and practical pre-visit instructions. Keep the tone professional but...
[input]: Patient symptoms: Hello , My name is Brittney I I am 12 years old . Today I have discovered a limp on the left side of my neck , about the size of a large pea . I ve been looking for answers but I hav...
[output]: Confirmation: Your appointment with Dr. Daniel Walsh in General Medicine has been confirmed for Wednesday at 11:00. We're here to help with your concerns.
Instructions: Hi,Nothing to worry, this lump ...


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def format_chat(example):
    """Convert instruction/input/output into LLaMA-3 chat format."""
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_dataset = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
test_dataset = dataset["test"].map(format_chat, remove_columns=dataset["test"].column_names)

print(f"Formatted sample (first 500 chars):\n{train_dataset[0]['text'][:500]}")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Map:   0%|          | 0/16604 [00:00<?, ? examples/s]

Map:   0%|          | 0/2931 [00:00<?, ? examples/s]

Formatted sample (first 500 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 22 Mar 2026

You are a compassionate medical assistant. A patient has been assigned an appointment. Write a warm, clear appointment confirmation and practical pre-visit instructions. Keep the tone professional but reassuring. Format your response as:
Confirmation: <one sentence confirming the appointment>
Instructions: <2-4 specific pre-visit instructions><|eot_id|><|start_header_id|>us


In [11]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False
print(f"Model loaded: {MODEL_ID} ({model.num_parameters() / 1e9:.2f}B parameters)")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded: meta-llama/Llama-3.2-3B-Instruct (3.21B parameters)


In [12]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [13]:
from trl import SFTTrainer, SFTConfig

In [14]:
import math
total_steps = math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM)) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

training_args = SFTConfig(
    output_dir=DRIVE_CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=warmup_steps,
    bf16=True,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
)

Map:   0%|          | 0/16604 [00:00<?, ? examples/s]

Map:   0%|          | 0/2931 [00:00<?, ? examples/s]

In [16]:
import glob

# Check for checkpoints to resume from (after Colab disconnect)
checkpoints = sorted(glob.glob(f"{DRIVE_CKPT_DIR}/checkpoint-*"))

if checkpoints:
    last_ckpt = checkpoints[-1]
    print(f"Found {len(checkpoints)} checkpoint(s) on Drive!")
    print(f"Resuming from: {last_ckpt}")
    train_result = trainer.train(resume_from_checkpoint=last_ckpt)
else:
    print("No checkpoints found. Starting fresh training...")
    train_result = trainer.train()

print(f"\n{'='*50}\nTraining complete!")
print(f"  Final Train loss: {train_result.training_loss:.4f}")
print(f"  Total steps: {train_result.global_step}")

No checkpoints found. Starting fresh training...


Step,Training Loss,Validation Loss
500,1.119431,1.142138
1000,1.050657,1.137778
1500,0.952059,1.147820
2000,0.942554,1.138718



Training complete!
  Final Train loss: 1.0110
  Total steps: 2076


In [17]:
ADAPTER_DIR = f"{DRIVE_CKPT_DIR}/final_adapter"

print(f"Saving final adapter to Drive...")
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import subprocess
result = subprocess.run(["du", "-sh", ADAPTER_DIR], capture_output=True, text=True)
print(f"Adapter saved to: {ADAPTER_DIR}")
print(f"Size: {result.stdout.strip()}")

Saving final adapter to Drive...
Adapter saved to: /content/drive/MyDrive/response_generator/final_adapter
Size: 110M	/content/drive/MyDrive/response_generator/final_adapter


In [18]:

eval_result = trainer.evaluate()
print(f"Eval loss: {eval_result['eval_loss']:.4f}")

Eval loss: 1.1378
